# Transform BOU_Transactions Data

1. Read bronze `transactions` table
1. Drop unwanted columns
1. Standardise column names using snake_case 
1. Filter out rows where `txn_id` is null (business key validation)
1. Remove duplicate records
1. Write the transformed data to silver `circuits` table

> Below changes are required to implement Incremental Load Processing
1. Accept batch_id as a parameter to the notebook
1. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
1. Add created_timestamp, updated_timestamp and batch_id to the silver table. 
1. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.
    - Ensure that we are not overwriting the data in silver table by older bronze data (re-run scenario)

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
import pyspark.sql.functions as F

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.BOU_transactions"
silver_table = f"{catalog_name}.{silver_schema}.BOU_transactions"

In [0]:
bou_df = spark.read.table("payment_app.bronze.bou_transactions").filter(F.col("batch_id")==v_batch_id)

In [0]:
display(bou_df)

In [0]:
bou_renamed = bou_df.withColumnsRenamed({"TxnID":"txn_id","BillerID":"biller_id",               "CustomerID":"customer_id","TxnAmount":"txn_amount","Status":"status","BillerRefID":"biller_ref_id","Timestamp":"transaction_time"})

In [0]:
bou_final = bou_renamed.filter(F.col("txn_id").isNotNull()).dropDuplicates()

In [0]:
bou_final = bou_final.withColumn("created_timestamp", F.current_timestamp()) \
                    .withColumn("updated_timestamp", F.current_timestamp())

In [0]:
bou_final.createOrReplaceTempView("vw_bou_final")

In [0]:
bou_final.printSchema()

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
        bou_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
else:
    spark.sql(f"""
    MERGE INTO {silver_table} AS tgt
    USING vw_bou_final AS src
    ON tgt.txn_id = src.txn_id

    WHEN MATCHED
    AND src.batch_id >= tgt.batch_id
    THEN UPDATE SET
        tgt.biller_id          = src.biller_id,
        tgt.customer_id        = src.customer_id,
        tgt.txn_amount         = src.txn_amount,
        tgt.status             = src.status,
        tgt.biller_ref_id      = src.biller_ref_id,
        tgt.transaction_time   = src.transaction_time,
        tgt.ingestion_timestamp = src.ingestion_timestamp,
        tgt.source_file        = src.source_file,
        tgt.batch_id          = src.batch_id,
        tgt.updated_timestamp = current_timestamp()

    WHEN NOT MATCHED
    THEN INSERT (
        txn_id,
        biller_id,
        customer_id,
        txn_amount,
        status,
        biller_ref_id,
        transaction_time,
        ingestion_timestamp,
        source_file,
        batch_id,
        created_timestamp,
        updated_timestamp
    )
    VALUES (
        src.txn_id,
        src.biller_id,
        src.customer_id,
        src.txn_amount,
        src.status,
        src.biller_ref_id,
        src.transaction_time,
        src.ingestion_timestamp,
        src.source_file,
        src.batch_id,
        src.created_timestamp,
        src.updated_timestamp
    )
    """)

In [0]:
display(spark.table(silver_table))